# CRNN + CTC — **Uncapped Receptive Field** Ablation

This notebook trains `CRNN_CTC_Uncapped`, the ablation variant of the
production `CRNN_CTC` model.

## What changes?
| Component | Original `CRNN_CTC` | `CRNN_CTC_Uncapped` (this) |
|---|---|---|
| 2D CNN Encoder | 4 blocks, identical | **identical** |
| Height Collapse | `mean(dim=2)` | **identical** |
| **1D Dilated CNN** | 4 blocks, dilations `[1,2,4,8]`, RF≈60 | **6 blocks, dilations `[1,2,4,8,16,32]`, RF≈252** |
| Classifier / CTC | Linear(256,11), blank=10 | **identical** |

## Hypothesis
With RF≈252 the uncapped model can see an entire training sequence
(≤12 digits ≈ 96 steps after 8× downsampling) in one receptive window.
This enables a pathological length prior. If the hypothesis holds, the
uncapped model will **fail to generalise to longer sequences** even though
it matches in-distribution accuracy.

## Data Pipeline
All dataset and utility code is **reused verbatim from `src/ctc/`** to
guarantee a fair, apples-to-apples comparison.

## 1 · Mount Google Drive & Create Checkpoint Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Checkpoint directory (separate from the baseline CRNN_CTC checkpoints) ──
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoint_CRNN_CTC_Uncapped"
METRICS_DIR    = os.path.join(CHECKPOINT_DIR, "metrics")
BEST_CKPT      = os.path.join(CHECKPOINT_DIR, "best_ctc_uncapped.pt")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR,    exist_ok=True)

print(f"Checkpoint directory : {CHECKPOINT_DIR}")
print(f"Best checkpoint path : {BEST_CKPT}")

## 2 · Clone Repository & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/AmineAitLaamim/digit-sequence-reader.git"
!git clone {REPO_URL}
%cd digit-sequence-reader
!pip install -r requirements.txt -q
!pip install albumentations opencv-python-headless -q

## 3 · Verify GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device       :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4 · Path Setup & Imports

We import the **new** `CRNN_CTC_Uncapped` model but reuse the **existing**
`src/ctc/` dataset and utilities unchanged — this is the key to a fair ablation.

In [ ]:
import sys, os

# Add the repo root so both src.CRNN_CTC_Uncapped and src.ctc are importable.
ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# ── New ablation model ───────────────────────────────────────────────────────
from src.CRNN_CTC_Uncapped.model import CRNN_CTC_Uncapped, greedy_decode

# ── Reusing existing dataset/utils to maintain DRY principles and ensure an
#    identical data pipeline for fair ablation. ──────────────────────────────
from src.ctc.dataset import (
    InfiniteCTCDataset,
    collate_fn          as ctc_collate_fn,
    get_dataloaders,
    build_multidigit_bank,
    get_digit_aug_pipeline,
    make_sequence,
)
from src.ctc.config import config   # shared config dict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import csv

print("All imports successful ✔")

## 5 · Hyperparameters

> **Ablation note — short training sequences on purpose.**  
> We deliberately cap `max_length=12` here (same as the baseline `CRNN_CTC`
> run).  After training, we will extrapolate to lengths 20, 30, 50 … to test
> whether the uncapped receptive field causes a length prior.

In [ ]:
# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE   = 64
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 30
CLIP_GRAD    = 1.0
NUM_WORKERS  = 2

# ── Data / curriculum settings ───────────────────────────────────────────────
# IMPORTANT: Keep training short (max_length=12) to set up the length
# extrapolation test.  This mirrors the baseline training exactly.
TRAIN_MAX_LENGTH = 12    # same cap used for baseline CRNN_CTC
TRAIN_SIZE       = 100_000
VAL_SIZE         = 10_000

# Override relevant config keys so the shared dataset code picks them up.
config['batch_size']       = BATCH_SIZE
config['max_seq_len_final'] = TRAIN_MAX_LENGTH
config['train_size']        = TRAIN_SIZE
config['val_size']          = VAL_SIZE
config['num_workers']       = NUM_WORKERS

# LR scheduler
LR_PATIENCE = 5
LR_FACTOR   = 0.5
LR_MIN      = 1e-6

# Early stopping
EARLY_STOP_PATIENCE = 12

print("Hyperparameters set ✔")
print(f"  Batch size        : {BATCH_SIZE}")
print(f"  Learning rate     : {LR}")
print(f"  Epochs            : {EPOCHS}")
print(f"  Max train length  : {TRAIN_MAX_LENGTH}  (short — for extrapolation test)")

## 6 · Optional Shape Sanity-Check

In [ ]:
!python -m src.CRNN_CTC_Uncapped.model

## 7 · Build Digit Bank & Data Loaders

In [ ]:
# Reusing existing get_dataloaders() — identical to the baseline run.
print("Building digit bank and validation loader (first run downloads datasets)...")
digit_bank, val_loader, _ = get_dataloaders(data_path=config['data_path'])

STEPS_PER_EPOCH = TRAIN_SIZE // BATCH_SIZE
print(f"Steps per epoch : {STEPS_PER_EPOCH}")

## 8 · Model Initialisation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device : {device}")

# Instantiate the ABLATION model (not the baseline CRNN_CTC).
model = CRNN_CTC_Uncapped().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters      : {n_params:,}")
print(f"Architecture    : 6-block 1D Dilated CNN, dilations [1,2,4,8,16,32], RF≈252 steps")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=LR_PATIENCE,
    factor=LR_FACTOR,
    min_lr=LR_MIN,
)

# ── Optional: resume from an existing checkpoint ─────────────────────────────
start_epoch  = 1
best_val_seq = 0.0

if os.path.exists(BEST_CKPT):
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    except Exception:
        pass
    start_epoch  = ckpt['epoch'] + 1
    best_val_seq = ckpt.get('val_seq_acc', 0.0)
    print(f"Resumed from epoch {ckpt['epoch']} | val_seq_acc={best_val_seq:.4f}")
else:
    print("No existing checkpoint — training from scratch.")

## 9 · Helper Functions

Metric computation uses the same Levenshtein-based CER as the baseline run.

In [ ]:
# ── Levenshtein distance (CER numerator) ─────────────────────────────────────
def _levenshtein(a, b):
    """Standard DP edit distance."""
    if len(a) < len(b):
        a, b = b, a
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb))
        prev = cur
    return prev[-1]


def compute_metrics(decoded, target_lengths, targets_flat):
    """Returns (seq_acc, cer) for a batch."""
    B = len(decoded)
    seq_correct   = 0
    edit_distance = 0
    total_chars   = 0
    cursor = 0
    for i, length in enumerate(target_lengths.tolist()):
        gt   = targets_flat[cursor: cursor + length].tolist()
        pred = decoded[i]
        cursor += length
        if pred == gt:
            seq_correct += 1
        edit_distance += _levenshtein(pred, gt)
        total_chars   += length
    return seq_correct / B, edit_distance / max(1, total_chars)


def save_checkpoint(model, optimizer, epoch, val_loss, val_seq_acc, path):
    """Save model + optimizer state to disk."""
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss':             val_loss,
        'val_seq_acc':          val_seq_acc,
    }, path)


@torch.no_grad()
def validate(model, dataloader, device):
    """Run a full validation pass. Returns (avg_loss, seq_acc, char_acc)."""
    model.eval()
    total_loss, total_seq, total_cer, n = 0., 0., 0., 0
    for batch in dataloader:
        images         = batch['images'].to(device)
        targets        = batch['targets'].to(device)
        target_lengths = batch['target_lengths'].to(device)
        loss, logits   = model(images, targets=targets, target_lengths=target_lengths)
        total_loss    += loss.item()
        decoded        = greedy_decode(logits)            # reuse greedy_decode from model.py
        seq_acc, cer   = compute_metrics(decoded, target_lengths, targets)
        total_seq     += seq_acc
        total_cer     += cer
        n             += 1
    return total_loss / n, total_seq / n, 1.0 - total_cer / n


print("Helper functions defined ✔")

## 10 · Training Loop

Identical structure to the baseline `src/ctc/train.py`:
- Training dataset rebuilt each epoch so augmentation intensity can follow the curriculum.
- Best checkpoint saved to Drive based on **validation sequence accuracy**.
- Gradient clipping at `CLIP_GRAD`.
- Early stopping after `EARLY_STOP_PATIENCE` epochs without improvement.

In [ ]:
from tqdm import tqdm

# ── CSV metrics file ──────────────────────────────────────────────────────────
metrics_file = os.path.join(METRICS_DIR, 'ctc_uncapped_metrics.csv')
if not os.path.exists(metrics_file):
    with open(metrics_file, 'w', newline='') as f:
        csv.writer(f).writerow([
            'epoch', 'train_loss', 'val_loss',
            'train_seq_acc', 'val_seq_acc',
            'train_char_acc', 'val_char_acc', 'lr',
        ])

LOG_EVERY          = 50   # compute metrics every N steps (rest: loss only)
early_stop_counter = 0

# ─────────────────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    # Train dataset rebuilt each epoch so aug curriculum can update.
    # Reusing InfiniteCTCDataset from src/ctc/dataset.py — identical to baseline.
    train_ds = InfiniteCTCDataset(
        digit_bank, config, size=TRAIN_SIZE, augment=True, epoch=epoch)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        collate_fn=ctc_collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )

    # ── Train one epoch ───────────────────────────────────────────────────────
    model.train()
    total_loss, total_seq, total_cer, n_metric_steps = 0., 0., 0., 0

    pbar = tqdm(enumerate(train_loader), total=STEPS_PER_EPOCH,
                desc="  train", leave=False)
    for step, batch in pbar:
        if step >= STEPS_PER_EPOCH:
            break

        images         = batch['images'].to(device)
        targets        = batch['targets'].to(device)
        target_lengths = batch['target_lengths'].to(device)

        optimizer.zero_grad()
        loss, logits = model(images, targets=targets, target_lengths=target_lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % LOG_EVERY == 0 or (step + 1) == STEPS_PER_EPOCH:
            with torch.no_grad():
                decoded  = greedy_decode(logits.detach())
                seq_acc, cer = compute_metrics(decoded, target_lengths, targets)
            total_seq += seq_acc
            total_cer += cer
            n_metric_steps += 1
            pbar.set_postfix(
                loss=f"{loss.item():.3f}",
                seq=f"{seq_acc:.3f}",
                cer=f"{cer:.3f}",
            )

    train_loss    = total_loss / STEPS_PER_EPOCH
    train_seq_acc = total_seq  / max(1, n_metric_steps)
    train_char_acc = 1.0 - total_cer / max(1, n_metric_steps)

    # ── Validate ──────────────────────────────────────────────────────────────
    val_loss, val_seq_acc, val_char_acc = validate(model, val_loader, device)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"  Train | loss={train_loss:.4f}  seq_acc={train_seq_acc:.4f}  char_acc={train_char_acc:.4f}")
    print(f"  Val   | loss={val_loss:.4f}  seq_acc={val_seq_acc:.4f}  char_acc={val_char_acc:.4f}  lr={current_lr}")

    scheduler.step(val_loss)

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if val_seq_acc > best_val_seq:
        best_val_seq       = val_seq_acc
        early_stop_counter = 0
        save_checkpoint(model, optimizer, epoch, val_loss, val_seq_acc, BEST_CKPT)
        print(f"  ✔ New best model saved (val_seq_acc={val_seq_acc:.4f})")
    else:
        early_stop_counter += 1

    # ── Log metrics ───────────────────────────────────────────────────────────
    with open(metrics_file, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch, train_loss, val_loss,
            train_seq_acc, val_seq_acc,
            train_char_acc, val_char_acc, current_lr,
        ])

    if early_stop_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs.")
        break

print("\nTraining complete. Best val_seq_acc:", best_val_seq)

## 11 · Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(metrics_file)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('CRNN_CTC_Uncapped — Ablation Training Curves', fontsize=13)

axes[0].plot(df['epoch'], df['train_loss'], label='train')
axes[0].plot(df['epoch'], df['val_loss'],   label='val')
axes[0].set_title('CTC Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(df['epoch'], df['train_seq_acc'], label='train')
axes[1].plot(df['epoch'], df['val_seq_acc'],   label='val')
axes[1].set_title('Sequence Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(df['epoch'], df['train_char_acc'], label='train')
axes[2].plot(df['epoch'], df['val_char_acc'],   label='val')
axes[2].set_title('Character Accuracy (1 − CER)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.show()

## 12 · Length-Extrapolation Test (the critical ablation result)

Load the best checkpoint and evaluate on sequences **longer than the training
maximum** (`max_length=12`).  Compare these numbers against the baseline
`CRNN_CTC` results to see whether the uncapped receptive field degrades
out-of-distribution performance.

Expected result (if hypothesis holds):
- `L ≤ 12` : similar accuracy to baseline ✅
- `L > 12` : significantly lower accuracy than baseline ❌  ← the "failure mode"

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt  = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_seq_acc={ckpt['val_seq_acc']:.4f})")

# ── Build a clean (no-aug) digit bank for deterministic evaluation ────────────
bank      = build_multidigit_bank(config['data_path'])
clean_aug = get_digit_aug_pipeline(augment=False, config=config)

# ── Evaluate at multiple lengths ──────────────────────────────────────────────
N_SAMPLES = 200   # samples per length — more → lower variance

print(f"\n{'L':>4}  {'seq_acc':>8}  {'CER':>8}")
print("-" * 28)

for L in [3, 5, 7, 10, 12, 15, 20, 25, 30]:
    correct = 0
    total_ed, total_chars = 0, 0

    # Temporarily override config to force sequences of length L.
    saved_min, saved_max = config['min_seq_len'], config['max_seq_len']
    config['min_seq_len'] = config['max_seq_len'] = L

    for _ in range(N_SAMPLES):
        img, true_digits = make_sequence(bank, clean_aug, config, augment=False, epoch=1)
        with torch.no_grad():
            logits = model(img.unsqueeze(0).to(device))   # [1, T, 11]
        pred = greedy_decode(logits)[0]

        if pred == true_digits:
            correct += 1
        total_ed    += _levenshtein(pred, true_digits)
        total_chars += L

    config['min_seq_len'] = saved_min
    config['max_seq_len'] = saved_max

    seq_acc = correct / N_SAMPLES
    cer     = total_ed / max(1, total_chars)
    ood_tag = "  ← OOD" if L > 12 else ""
    print(f"{L:>4}  {seq_acc:>8.3f}  {cer:>8.3f}{ood_tag}")

print("\nDone. Compare OOD rows against the baseline CRNN_CTC results.")

## 13 · Inference on a Custom Image

In [ ]:
from google.colab import files

uploaded = files.upload()
if uploaded:
    img_path = list(uploaded.keys())[0]
    # Reuse the baseline inference script — same entry-point, different checkpoint.
    !python -m src.ctc.inference \
        --image "{img_path}" \
        --checkpoint "{BEST_CKPT}" \
        --visualize